# Instacart Transformation Pipeline

In [1]:
%idle_timeout 2880
%glue_version 5.1
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.1
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: 23745f29-248d-4e0f-a8a1-8569ba2f206e
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 23745f29-248d-4e0f-a8a1-8569ba2f206e to get into ready status...
Session 23745f29-248d-4e0f-a8a1-8569ba2f206e 

In [ ]:
from pyspark.sql.functions import *

In [28]:
df_orders = spark.read.option("header","true").option("inferSchema", "true").csv("s3://instacart-data-engineering/data/raw-data/orders.csv")
df_orders.show(5)

+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
| 2539329|      1|   prior|           1|        2|                8|                  NULL|
| 2398795|      1|   prior|           2|        3|                7|                  15.0|
|  473747|      1|   prior|           3|        3|               12|                  21.0|
| 2254736|      1|   prior|           4|        4|                7|                  29.0|
|  431534|      1|   prior|           5|        4|               15|                  28.0|
+--------+-------+--------+------------+---------+-----------------+----------------------+
only showing top 5 rows


In [29]:
df_products = spark.read.option("header","true").option("inferSchema", "true").csv("s3://instacart-data-engineering/data/raw-data/products.csv")
df_products.show(5)

+----------+--------------------+--------+-------------+
|product_id|        product_name|aisle_id|department_id|
+----------+--------------------+--------+-------------+
|         1|Chocolate Sandwic...|      61|           19|
|         2|    All-Seasons Salt|     104|           13|
|         3|Robust Golden Uns...|      94|            7|
|         4|Smart Ones Classi...|      38|            1|
|         5|Green Chile Anyti...|       5|           13|
+----------+--------------------+--------+-------------+
only showing top 5 rows


In [38]:
df_order_products__prior = spark.read.option("header","true").option("inferSchema", "true").csv("s3://instacart-data-engineering/data/raw-data/order_products__prior.csv")
df_order_products__prior.show(5)
df_order_products__prior.count()

+--------+----------+-----------------+---------+
|order_id|product_id|add_to_cart_order|reordered|
+--------+----------+-----------------+---------+
|       2|     33120|                1|        1|
|       2|     28985|                2|        1|
|       2|      9327|                3|        0|
|       2|     45918|                4|        1|
|       2|     30035|                5|        0|
+--------+----------+-----------------+---------+
only showing top 5 rows

32434489


In [39]:
df_order_products__train = spark.read.option("header","true").option("inferSchema", "true").csv("s3://instacart-data-engineering/data/raw-data/order_products__train.csv")
df_order_products__train.show(5)
df_order_products__train.count()

+--------+----------+-----------------+---------+
|order_id|product_id|add_to_cart_order|reordered|
+--------+----------+-----------------+---------+
|       1|     49302|                1|        1|
|       1|     11109|                2|        1|
|       1|     10246|                3|        0|
|       1|     49683|                4|        0|
|       1|     43633|                5|        1|
+--------+----------+-----------------+---------+
only showing top 5 rows

1384617


In [25]:
df_aisles=spark.read.option("header","true").option("inferSchema", "true").csv("s3://instacart-data-engineering/data/raw-data/aisles.csv")
df_aisles.show(5)

+--------+--------------------+
|aisle_id|               aisle|
+--------+--------------------+
|       1|prepared soups sa...|
|       2|   specialty cheeses|
|       3| energy granola bars|
|       4|       instant foods|
|       5|marinades meat pr...|
+--------+--------------------+
only showing top 5 rows


In [26]:
df_departments=spark.read.option("header","true").option("inferSchema", "true").csv("s3://instacart-data-engineering/data/raw-data/departments.csv")
df_departments.show(5)

+-------------+----------+
|department_id|department|
+-------------+----------+
|            1|    frozen|
|            2|     other|
|            3|    bakery|
|            4|   produce|
|            5|   alcohol|
+-------------+----------+
only showing top 5 rows


In [32]:
df_orders.printSchema()
df_products.printSchema()
df_order_products__prior.printSchema()
df_order_products__train.printSchema()
df_aisles.printSchema()
df_departments.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: integer (nullable = true)
 |-- order_dow: integer (nullable = true)
 |-- order_hour_of_day: integer (nullable = true)
 |-- days_since_prior_order: double (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: string (nullable = true)
 |-- department_id: string (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)

root
 |-- aisle_id: integer (nullable = true)
 |-- aisle: string (nullable = true)

root
 |-- department_id: integer (nullable = true)
 |-- 

In [ ]:
df_products = (
    df_products
    .withColumn("aisle_id", col("aisle_id").cast("int"))
    .withColumn("department_id", col("department_id").cast("int"))
)

In [46]:
df_products.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)


In [37]:
df_order_products = df_order_products__prior.unionByName(df_order_products__train)
df_order_products.show()

+--------+----------+-----------------+---------+
|order_id|product_id|add_to_cart_order|reordered|
+--------+----------+-----------------+---------+
|       2|     33120|                1|        1|
|       2|     28985|                2|        1|
|       2|      9327|                3|        0|
|       2|     45918|                4|        1|
|       2|     30035|                5|        0|
|       2|     17794|                6|        1|
|       2|     40141|                7|        1|
|       2|      1819|                8|        1|
|       2|     43668|                9|        0|
|       3|     33754|                1|        1|
|       3|     24838|                2|        1|
|       3|     17704|                3|        1|
|       3|     21903|                4|        1|
|       3|     17668|                5|        1|
|       3|     46667|                6|        1|
|       3|     17461|                7|        1|
|       3|     32665|                8|        1|


In [40]:
df_order_products.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)


In [42]:
df_orders = df_orders.withColumn(
    "is_weekend",
    when(col("order_dow").isin([0, 6]), 1).otherwise(0)
)

df_orders = df_orders.withColumn(
    "time_of_day",
    when(col("order_hour_of_day") < 12, "Morning")
    .when(col("order_hour_of_day") < 17, "Afternoon")
    .when(col("order_hour_of_day") < 21, "Evening")
    .otherwise("Night")
)

df_orders = df_orders.withColumn(
    "days_since_prior_order",
    coalesce(col("days_since_prior_order"), lit(0))
)

df_orders.show(30)

+--------+-------+--------+------------+---------+-----------------+----------------------+----------+-----------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|is_weekend|time_of_day|
+--------+-------+--------+------------+---------+-----------------+----------------------+----------+-----------+
| 2539329|      1|   prior|           1|        2|                8|                   0.0|         0|    Morning|
| 2398795|      1|   prior|           2|        3|                7|                  15.0|         0|    Morning|
|  473747|      1|   prior|           3|        3|               12|                  21.0|         0|  Afternoon|
| 2254736|      1|   prior|           4|        4|                7|                  29.0|         0|    Morning|
|  431534|      1|   prior|           5|        4|               15|                  28.0|         0|  Afternoon|
| 3367565|      1|   prior|           6|        2|                7|            

In [43]:
from pyspark.sql.functions import when, col,coalesce, lit
df_order_products = df_order_products.withColumn(
    "reorder_status",
    when(col("reordered") == 1, "Reordered")
    .otherwise("First Purchase")
)

df_order_products.show()

+--------+----------+-----------------+---------+--------------+
|order_id|product_id|add_to_cart_order|reordered|reorder_status|
+--------+----------+-----------------+---------+--------------+
|       2|     33120|                1|        1|     Reordered|
|       2|     28985|                2|        1|     Reordered|
|       2|      9327|                3|        0|First Purchase|
|       2|     45918|                4|        1|     Reordered|
|       2|     30035|                5|        0|First Purchase|
|       2|     17794|                6|        1|     Reordered|
|       2|     40141|                7|        1|     Reordered|
|       2|      1819|                8|        1|     Reordered|
|       2|     43668|                9|        0|First Purchase|
|       3|     33754|                1|        1|     Reordered|
|       3|     24838|                2|        1|     Reordered|
|       3|     17704|                3|        1|     Reordered|
|       3|     21903|    

In [47]:
df_product_details = (
    df_products
    .join(df_aisles, "aisle_id", "left")
    .join(df_departments, "department_id", "left")
)

df_product_details.show()

+-------------+--------+----------+--------------------+--------------------+---------------+
|department_id|aisle_id|product_id|        product_name|               aisle|     department|
+-------------+--------+----------+--------------------+--------------------+---------------+
|           19|      61|         1|Chocolate Sandwic...|       cookies cakes|         snacks|
|           13|     104|         2|    All-Seasons Salt|   spices seasonings|         pantry|
|            7|      94|         3|Robust Golden Uns...|                 tea|      beverages|
|            1|      38|         4|Smart Ones Classi...|        frozen meals|         frozen|
|           13|       5|         5|Green Chile Anyti...|marinades meat pr...|         pantry|
|           11|      11|         6|        Dry Nose Oil|    cold flu allergy|  personal care|
|            7|      98|         7|Pure Coconut Wate...|       juice nectars|      beverages|
|            1|     116|         8|Cut Russet Potato...|    

In [48]:
df_product_details.printSchema()

root
 |-- department_id: integer (nullable = true)
 |-- aisle_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle: string (nullable = true)
 |-- department: string (nullable = true)


In [50]:
df_orders.write.mode("overwrite").parquet("s3://instacart-data-engineering/data/transformed-data/orders/")

In [51]:
df_products.write.mode("overwrite").parquet("s3://instacart-data-engineering/data/transformed-data/products/")

In [52]:
df_order_products.write.mode("overwrite").parquet("s3://instacart-data-engineering/data/transformed-data/order_products/")

In [53]:
df_product_details.write.mode("overwrite").parquet("s3://instacart-data-engineering/data/transformed-data/product_details/")

In [54]:
df_aisles.write.mode("overwrite").parquet("s3://instacart-data-engineering/data/transformed-data/aisles/")

In [55]:
df_departments.write.mode("overwrite").parquet("s3://instacart-data-engineering/data/transformed-data/departments/")